In [1]:
# Important for Colab
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Flausch detection: Task 2 span classification -- Train BERT Token Classifier  

### Dataset preparation

In [2]:
import pandas as pd
import os
import sys
import numpy as np

# work with the train data only
#path = ""
path = "Input/Data/train/"
data = pd.read_json(path + "train_task1.json")
data2 = pd.read_json(path + "train_task2.json")


In [3]:
# set checkpoint and load tokenizer
from transformers import AutoTokenizer
checkpoint = "deepset/gbert-large"
model_name = checkpoint.split("/")[-1]
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


In [4]:
# filter by flausch=yes for task 2
#mode = "only_flausch"
mode = "all"
if mode =="only_flausch":
    dataflausch = data[data["flausch"]=="yes"].copy().reset_index(drop=True)
if mode =="all":
    dataflausch = data.copy().reset_index(drop=True)

In [5]:
# prüfe wie gut unser test/train split ist: 
all1 = pd.read_csv(path + "task1.csv")
all2 = pd.read_csv(path + "task2.csv")
allflausch = all1[all1["flausch"]=="yes"]
print("ratio flausch in all data", allflausch.shape[0]/all1.shape[0])
print("ratio flausch in train data", dataflausch.shape[0]/data.shape[0])
print("flausch / #spans in all data", allflausch.shape[0]/all2.shape[0])
print("flausch / #spans in train data", dataflausch.shape[0]/data2.shape[0])
print("similar distributions")



ratio flausch in all data 0.2907143049896106
ratio flausch in train data 1.0
flausch / #spans in all data 0.6818785999113868
flausch / #spans in train data 2.335177146057975
similar distributions


In [6]:
# generate BIO tag scheme (begin/inside/outside)
# dictionaries to map between tag ids and tag names
tags = sorted(data2["type"].unique())
tag2id = {'O': 0}
for i, tag in enumerate(tags):
    tag2id["I-"+ tag] = 2*i +1
    tag2id["B-"+ tag] = 2*(i+1)

id2tag = {v: k for k, v in tag2id.items()}


In [7]:
#some testing
tokenized = tokenizer(["Das ist ein Horrorfilm!"], return_offsets_mapping=True)
print(tokenized.encodings)
print(tokenized.encodings[0].tokens)
print(tokenized.encodings[0].ids)
print(tokenized.encodings[0].offsets)


[Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]
['[CLS]', 'Das', 'ist', 'ein', 'Horror', '##film', '!', '[SEP]']
[102, 347, 215, 143, 28816, 5241, 3330, 103]
[(0, 0), (0, 3), (4, 7), (8, 11), (12, 18), (18, 22), (22, 23), (0, 0)]


In [8]:
# main function that creates target token sequences for tokenized training data

def align_labels_with_tokens(tokenizer, data, tag2id):
    # tokenize text and get offset_mapping
    text = data["comment"]
    types = data["types"]  # list of types of the flausch spans
    span_pairs = data["span_pairs"]  # list of boundaries of labeled flausch spans
    tokenized_input = tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=512)
    # encodings[0] für den einzelnen Text
    encoding = tokenized_input.encodings[0]

    token_ids = encoding.ids  # Token-IDs
    label_ids = [tag2id['O'] for i in  range(len(encoding.tokens))]  # Initialisiere mit O-Labels für jedes Wort/Token


    # Iteriere über jeden Span, der gelabelt werden soll
    for i in range(len(span_pairs)):
        span_start_char = span_pairs[i][0]
        span_end_char = span_pairs[i][1]
        span_type = types[i]

        # Finde die Token-Indizes, die diesen Span abdecken
        token_start_idx = encoding.char_to_token(span_start_char)
        token_end_idx = encoding.char_to_token(span_end_char - 1) # end_char ist exklusiv

        # Wenn der Span nicht durch die Tokenisierung abgedeckt wird (z.B. wegen Truncation)
        if token_start_idx is None or token_end_idx is None:
            continue

        # Wenn der Span über mehrere Tokens geht oder ein einzelnes Token ist
        for current_token_idx in range(token_start_idx, token_end_idx + 1):
            if current_token_idx == token_start_idx:
                # Dies ist das erste Token, das den Span abdeckt -> B-Tag
                label_ids[current_token_idx] = tag2id["B-"+span_type]
            else:
                # Alle nachfolgenden Tokens, die denselben Span abdecken -> I-Tag
                # Alle Tokens vom Start bis Ende des Spans sind entweder B oder I.
                # Das erste ist B, der Rest I.

                # Wenn token_start_idx == token_end_idx (einzelnes Token für Span), ist es B.
                # Wenn token_start_idx != token_end_idx, dann erstes ist B, der Rest ist I.
                label_ids[current_token_idx] = tag2id["I-"+ span_type]

    return tokenized_input["input_ids"], label_ids



In [11]:
i=4
input_ids, label_ids = align_labels_with_tokens(tokenizer, dataflausch.loc[i], tag2id)
ex_spans = dataflausch["spans"][i]

print(ex_spans)
for idx,j in enumerate(label_ids):
    if j != 0:
        print(id2tag[j], tokenizer.decode(input_ids[idx]))

['wie wunderschön du in diesem video bist 3']
B-compliment wie
I-compliment wunderschön
I-compliment du
I-compliment in
I-compliment diesem
I-compliment v
I-compliment ##ideo
I-compliment bist
I-compliment 3


In [12]:
import pandas as pd
from datasets import Dataset, Features, Value, ClassLabel, Sequence
from sklearn.model_selection import train_test_split # Für den Split in Train/Test



all_input_ids = []
all_labels = []

print("Beginne mit der Vorverarbeitung des DataFrames...")
for idx, row in dataflausch.iterrows():
    input_ids_for_text, labels_for_text = align_labels_with_tokens(tokenizer, row, tag2id)

    # Sammle die Ergebnisse
    all_input_ids.append(input_ids_for_text)
    all_labels.append(labels_for_text)

    if (idx + 1) % 1000 == 0: # Fortschrittsanzeige alle 1000 Texte
        print(f"Verarbeitete {idx + 1}/{len(dataflausch)} Texte.")

print("Alle Texte erfolgreich vorverarbeitet.")
print(f"Anzahl vorbereiteter Beispiele: {len(all_input_ids)}")

max_label_id = max(id2tag.keys())
id2tag_ordered_list = [None] * (max_label_id + 1) # Erstelle eine Liste der richtigen Größe
for k, v in id2tag.items():
    id2tag_ordered_list[k] = v

features = Features({
    'input_ids': Sequence(Value('int32')), # Sequenz von Ganzzahlen für Token-IDs
    'labels': Sequence(ClassLabel(names=id2tag_ordered_list)) # Sequenz von ClassLabels für NER-Tags
})

# Erstelle ein Dictionary, das die Daten enthält
dataset_dict = {
    'input_ids': all_input_ids,
    'labels': all_labels
}

# Erstelle ein Dataset aus dem Dictionary und den definierten Features
prepared_dataset = Dataset.from_dict(dataset_dict, features=features)

print("\nHugging Face Dataset erfolgreich erstellt.")
print(prepared_dataset)
print("\nBeispiel des ersten Eintrags im Dataset:")
print(prepared_dataset[0])

# Überprüfe den ersten Eintrag noch einmal visuell
first_text = dataflausch["comment"].iloc[0] # .iloc[0] für den ersten Eintrag im DF
first_input_ids = prepared_dataset[0]['input_ids']
first_labels_ids = prepared_dataset[0]['labels']

print(f"\nOriginaltext (erster Eintrag): '{first_text}'")
print(f"Token (decoded): {[tokenizer.decode(t) for t in first_input_ids]}")
print(f"Labels (decoded): {[id2tag[l] for l in first_labels_ids]}")


# Teile das Dataset in Trainings- und Devsets auf (85% Training, 15% Development)
# Remember the current dataset is 90% of the original one. 10% test data have been already set aside
split_dataset = prepared_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split_dataset['train']
dev_dataset = split_dataset['test']

# create one datasetdict for train, dev and test
from datasets import DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset,
    'dev': dev_dataset,
})

print(f"\nTrainingsdatensatzgröße: {len(train_dataset)}")
print(f"Entwicklungsdatensatzgröße: {len(dev_dataset)}")



Beginne mit der Vorverarbeitung des DataFrames...
Verarbeitete 1000/33351 Texte.
Verarbeitete 2000/33351 Texte.
Verarbeitete 3000/33351 Texte.
Verarbeitete 4000/33351 Texte.
Verarbeitete 5000/33351 Texte.
Verarbeitete 6000/33351 Texte.
Verarbeitete 7000/33351 Texte.
Verarbeitete 8000/33351 Texte.
Verarbeitete 9000/33351 Texte.
Verarbeitete 10000/33351 Texte.
Verarbeitete 11000/33351 Texte.
Verarbeitete 12000/33351 Texte.
Verarbeitete 13000/33351 Texte.
Verarbeitete 14000/33351 Texte.
Verarbeitete 15000/33351 Texte.
Verarbeitete 16000/33351 Texte.
Verarbeitete 17000/33351 Texte.
Verarbeitete 18000/33351 Texte.
Verarbeitete 19000/33351 Texte.
Verarbeitete 20000/33351 Texte.
Verarbeitete 21000/33351 Texte.
Verarbeitete 22000/33351 Texte.
Verarbeitete 23000/33351 Texte.
Verarbeitete 24000/33351 Texte.
Verarbeitete 25000/33351 Texte.
Verarbeitete 26000/33351 Texte.
Verarbeitete 27000/33351 Texte.
Verarbeitete 28000/33351 Texte.
Verarbeitete 29000/33351 Texte.
Verarbeitete 30000/33351 Texte.

In [13]:
example = dataset_dict["train"][0]
example
pd.DataFrame([tokenizer.decode(x) for x in example["input_ids"]], [id2tag[x] for x in example["labels"]]).T#.rename(columns={0: 'Tokens', 1: 'Tags'})



,O,O,O,O,O,O,O,O,O,O,...,I-compliment,I-compliment,I-compliment,I-compliment,I-compliment,I-compliment,I-compliment,I-compliment,I-compliment,O
0,[CLS],ich,schl,##af,immer,in,viel,zu,großen,(,...,##ideo,so,schön,aus,!,!,*,-,*,[SEP]


In [14]:
dataset_dict["train"].features["labels"].feature.names

['O',
 'I-affection declaration',
 'B-affection declaration',
 'I-agreement',
 'B-agreement',
 'I-ambiguous',
 'B-ambiguous',
 'I-compliment',
 'B-compliment',
 'I-encouragement',
 'B-encouragement',
 'I-gratitude',
 'B-gratitude',
 'I-group membership',
 'B-group membership',
 'I-implicit',
 'B-implicit',
 'I-positive feedback',
 'B-positive feedback',
 'I-sympathy',
 'B-sympathy']

In [15]:
from collections import defaultdict, Counter

retrieved_id2tag_ordered_list = train_dataset.features['labels'].feature.names

# Initialisiere das Dictionary, um die Häufigkeiten zu speichern
split2freqs = defaultdict(Counter)


# Iteriere durch die Splits
for split_name, dataset_split in dataset_dict.items():
    # Iteriere durch jedes Beispiel im aktuellen Dataset-Split
    for example in dataset_split:
        label_ids_for_example = example['labels']
        for label_id in label_ids_for_example:
            # Konvertiere die numerische Label-ID zurück in den String-Tag
            tag_str = retrieved_id2tag_ordered_list[label_id]

            # Prüfe, ob es sich um einen "B-" Tag handelt
            if tag_str.startswith("B-"):
                # Extrahiere den eigentlichen Entitätstyp (z.B. "ORG" aus "B-ORG")
                tag_type = tag_str.split("-")[1]
                # Inkrementiere den Zähler für diesen Split und diesen Entitätstyp
                split2freqs[split_name][tag_type] += 1

# Erstelle ein Pandas DataFrame aus den gesammelten Häufigkeiten für eine bessere Übersicht
label_frequency_df = pd.DataFrame.from_dict(split2freqs, orient="index").fillna(0).astype(int)

label_frequency_df


,compliment,positive feedback,affection declaration,agreement,gratitude,encouragement,sympathy,implicit,group membership,ambiguous
train,2095,6505,2063,174,182,578,75,139,121,167
dev,403,1080,365,33,29,104,15,33,25,24


In [16]:
# huggingface login
from huggingface_hub import login
login()


In [ ]:
# dataset to huggingface

dataset_dict.push_to_hub("Wiebke/FlauschSpans", private=False)

Pushing dataset shards to the dataset hub:   0%|          | 0/1 [00:00<?, ?it/s]

Pushing dataset shards to the dataset hub:   0%|          | 0/1 [00:00<?, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


### Evaluation Metric



In [ ]:
dataset = dataset_dict

In [ ]:
!pip install seqeval
!pip install evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=614817bb4c1eb4e00b9625f0e8d930f571e59613ba778a0d65aceb6d6471476c
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.8 MB/s eta 0:00:00


In [ ]:
import evaluate
seqeval = evaluate.load("seqeval")

In [ ]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report


tags = dataset["train"].features['labels'].feature.names

tag2id = {name: i for i, name in enumerate(tags)}
id2tag = {i: name for i, name in enumerate(tags)}

label_list = list(tag2id.keys())

from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

# Annahme: Diese Variablen sind bereits GLOBAL in deinem Skript definiert:
# tags = dataset["train"].features['labels'].feature.names
# tag2id = {name: i for i, name in enumerate(tags)}
# id2tag = {i: name for i, name in enumerate(tags)}
# label_list = list(tag2id.keys()) # Dies ist die Liste aller deiner String-Labels (z.B. ['O', 'B-affection declaration', ...])

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Konvertierung der numerischen IDs zurück zu den Label-Strings
    # und Entfernen von Padding-Tokens (-100)
    true_labels = []
    for label_sequence in labels:
        true_labels.append([id2tag[l] for l in label_sequence if l != -100])

    true_predictions = []
    for prediction_sequence, label_sequence in zip(predictions, labels):
        true_predictions.append([
            id2tag[p] for p, l in zip(prediction_sequence, label_sequence) if l != -100
        ])

    # Berechne Metriken mit explizitem zero_division=0
    # Dies ist der Standard von seqeval und sorgt dafür, dass undefinierte Werte 0.0 werden.
    # Es hilft, die UndefinedMetricWarning zu kontrollieren.
    precision = precision_score(true_labels, true_predictions, zero_division=0)
    recall = recall_score(true_labels, true_predictions, zero_division=0)
    f1 = f1_score(true_labels, true_predictions, zero_division=0)

    # Generiere und gib den detaillierten Klassifikationsbericht aus.
    # Das `labels=label_list` Argument stellt sicher, dass alle deine definierten Labels
    # im Bericht erscheinen, auch wenn sie im aktuellen Batch nicht vorkommen.
    report = classification_report(
        true_labels,
        true_predictions,
        digits=4, # Anzahl der Dezimalstellen im Bericht
        zero_division=0, # Steuerung der 0-Division für den Bericht
        #labels=label_list # Wichtig, um alle Labels im Bericht zu haben führt aber zu Versionsproblemen
    )

    print("\n--- Classification Report (Aktueller Evaluationsschritt) ---")
    print(report)
    print("-----------------------------------------------------------")

    return {"precision": precision, "recall": recall, "f1": f1}




In [ ]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer)

### Training

In [ ]:
# load model with the correct head for token classification with the number of labels
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

num_labels = len(tag2id)  # Anzahl der Labels

model = AutoModelForTokenClassification.from_pretrained(
    checkpoint, num_labels=num_labels, id2label=id2tag, label2id=tag2id
)

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at deepset/gbert-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import pipeline
i=2
text = tokenizer.decode(dataset["dev"]["input_ids"][i])
classifier = pipeline("ner", model=model, tokenizer=tokenizer)

print(text)
for d in classifier(text):
  print(d["word"], d["entity"])

Device set to use cuda:0


[CLS] Das mit der Community - Aktion ist ne tolle Sache! Kannst du gerne öfter machen ; ) [SEP]
[CLS] B-encouragement
Das B-group membership
mit B-group membership
der B-group membership
Community B-group membership
- B-group membership
Aktion B-group membership
ist I-encouragement
ne B-group membership
tolle B-group membership
Sache B-group membership
! B-group membership
Kannst B-gratitude
du B-group membership
gerne B-group membership
öfter B-group membership
machen B-group membership
; B-group membership
) B-group membership
[SEP] B-gratitude


In [ ]:
from transformers import TrainingArguments, Trainer
import os

if mode == "all":
    modle_name = model_name + "_all"


# Define training parameters
training_args = TrainingArguments(
    output_dir="flausch_span_"+ model_name, # Verzeichnis für Modell, Checkpoints und Logs
    learning_rate=2e-5,
    metric_for_best_model="eval_f1", # competition ranking metric
    greater_is_better=True, # Setze auf False, wenn du Loss minimieren willst
    per_device_train_batch_size=16, # Reduziere um OOM zu vermeiden
    per_device_eval_batch_size=16,  # Reduziere um OOM zu vermeiden
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    logging_steps=500, # Alle 500 Schritte loggen
    fp16=True, # Für schnellere und speichereffizientere Berechnung auf GPU
    gradient_checkpointing=True, # Hilft gegen OOM bei großen Modellen, macht Training langsamer
    save_total_limit=1, # Nur das beste Modell speichern
    report_to="none",
    push_to_hub=False, # Zuerst trainieren, dann manuell pushen
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["dev"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics, # Eigene Funktion zur Metrikberechnung
)


<ipython-input-32-c452127e1583>:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
print("\nPredictions on dev data before training:")

# Die predict-Methode gibt ein Objekt zurück, das die Vorhersagen,
# die echten Labels und die Metriken enthält (falls der Trainer diese selbst berechnet hat).
predictions_output = trainer.predict(test_dataset=dataset["dev"])



Predictions on dev data before training:



--- Classification Report (Aktueller Evaluationsschritt) ---
                       precision    recall  f1-score   support

affection declaration     0.0000    0.0000    0.0000       367
            agreement     0.0000    0.0000    0.0000        31
            ambiguous     0.0000    0.0000    0.0000        24
           compliment     0.0000    0.0000    0.0000       384
        encouragement     0.0000    0.0000    0.0000        97
            gratitude     0.0000    0.0000    0.0000        32
     group membership     0.0000    0.0000    0.0000        20
             implicit     0.0000    0.0000    0.0000        34
    positive feedback     0.0000    0.0000    0.0000      1128
             sympathy     0.0000    0.0000    0.0000        15

            micro avg     0.0000    0.0000    0.0000      2132
            macro avg     0.0000    0.0000    0.0000      2132
         weighted avg     0.0000    0.0000    0.0000      2132

-----------------------------------------------------

In [ ]:

# Training
trainer.train()


Epoch,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1
1,0.588200,0.584174,0.005700,0.453898,0.628049,0.526958
2,0.387900,0.572010,0.005700,0.519428,0.664634,0.583128
3,0.256000,0.607512,0.005700,0.498549,0.644465,0.562193
4,0.147700,0.672550,0.005700,0.507452,0.654784,0.571780



--- Classification Report (Aktueller Evaluationsschritt) ---
                       precision    recall  f1-score   support

affection declaration     0.5249    0.6594    0.5845       367
            agreement     0.5667    0.5484    0.5574        31
            ambiguous     0.5833    0.2917    0.3889        24
           compliment     0.3182    0.6380    0.4246       384
        encouragement     0.4130    0.5876    0.4851        97
            gratitude     0.2951    0.5625    0.3871        32
     group membership     0.3333    0.4000    0.3636        20
             implicit     0.0286    0.0294    0.0290        34
    positive feedback     0.5300    0.6587    0.5874      1128
             sympathy     0.0588    0.0667    0.0625        15

            micro avg     0.4539    0.6280    0.5270      2132
            macro avg     0.3652    0.4442    0.3870      2132
         weighted avg     0.4701    0.6280    0.5325      2132

-----------------------------------------------------

TrainOutput(global_step=2068, training_loss=0.4116640017157136, metrics={'train_runtime': 797.1086, 'train_samples_per_second': 41.495, 'train_steps_per_second': 2.594, 'total_flos': 4339145973967290.0, 'train_loss': 0.4116640017157136, 'epoch': 4.0})

In [ ]:
text = tokenizer.decode(dataset["dev"]["input_ids"][i])
classifier = pipeline("ner", model=model, tokenizer=tokenizer)

print(text)
for d in classifier(text):
  print(d["word"], d["entity"])

Device set to use cuda:0


[CLS] Das mit der Community - Aktion ist ne tolle Sache! Kannst du gerne öfter machen ; ) [SEP]
Das B-positive feedback
mit I-positive feedback
der I-positive feedback
Community I-positive feedback
- I-positive feedback
Aktion I-positive feedback
ist I-positive feedback
ne I-positive feedback
tolle I-positive feedback
Sache I-positive feedback
! I-positive feedback
Kannst B-encouragement
du I-encouragement
gerne I-encouragement
öfter I-encouragement
machen I-encouragement
; I-encouragement
) I-encouragement


In [ ]:

print("\nPredictions on dev set after training.")

# Die predict-Methode gibt ein Objekt zurück, das die Vorhersagen,
# die echten Labels und die Metriken enthält (falls der Trainer diese selbst berechnet hat).
predictions_output = trainer.predict(test_dataset=dataset["dev"])


Predictions on dev set after training.



--- Classification Report (Aktueller Evaluationsschritt) ---
                       precision    recall  f1-score   support

affection declaration     0.5967    0.6975    0.6432       367
            agreement     0.5862    0.5484    0.5667        31
            ambiguous     0.4667    0.2917    0.3590        24
           compliment     0.4340    0.6250    0.5123       384
        encouragement     0.5169    0.6289    0.5674        97
            gratitude     0.3226    0.6250    0.4255        32
     group membership     0.6000    0.6000    0.6000        20
             implicit     0.0455    0.0294    0.0357        34
    positive feedback     0.5517    0.7101    0.6209      1128
             sympathy     0.0714    0.1333    0.0930        15

            micro avg     0.5194    0.6646    0.5831      2132
            macro avg     0.4192    0.4889    0.4424      2132
         weighted avg     0.5218    0.6646    0.5828      2132

-----------------------------------------------------

In [ ]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/Wiebke/flausch_span_gbert-large/commit/81b6614eaa43d1c336372550f358e6d7a58a4b48', commit_message='End of training', commit_description='', oid='81b6614eaa43d1c336372550f358e6d7a58a4b48', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Wiebke/flausch_span_gbert-large', endpoint='https://huggingface.co', repo_type='model', repo_id='Wiebke/flausch_span_gbert-large'), pr_revision=None, pr_num=None)